# 02 – Praproses, ROI, dan split (CP2, FR-02 sampai FR-04)

Lima langkah praproses → cache memmap uint8 (256×256, ber-hash) + peta ROI Frangi. Split per pasien (`splits.n_splits` lipatan × `splits.n_repeats` ulangan, 15% pasien latih sebagai validasi); split S7. Uji kebocoran L1 sampai L3, L5, L6.

Normalisasi ImageNet tetap (diterapkan saat dibaca), tidak ada citra dibuang, parameter Frangi dikunci setelah pemeriksaan visual.

**Notebook mandiri (Versi 2):** tanpa folder `src`, tanpa Docker. Semua kode ada di sel-sel notebook ini. Kode ditulis dari Versi 1 dan **belum dijalankan** di mesin penulis (tanpa torch/GPU); jalankan dari atas ke bawah di mesin RTX 3070 Ti. Hasil yang tidak dapat dihitung dilaporkan apa adanya, bukan diisi.

## 1. Pustaka tambahan (opsional)

In [ ]:
INSTALL = False      # True hanya sekali, di venv yang sama dengan notebook 00 (torch CUDA sudah terpasang di sana)

if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-image", "scikit-learn", "scipy", "pillow", "matplotlib", "pandas"])

## 2. Impor, konfigurasi, utilitas, pemeriksaan checkpoint, dan pengendalian kebocoran

In [ ]:
from __future__ import annotations

import contextlib, copy, datetime as dt, gc, hashlib, importlib, io, json, math, os, platform, random, re, shutil, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import yaml
from IPython.display import display, Image

# expandable_segments mengurangi fragmentasi memori, tetapi belum didukung di Windows.
if platform.system() != "Windows":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn
import torch.nn.functional as F


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "configs" / "base.yaml").exists():
            return q
    raise FileNotFoundError("configs/base.yaml tidak ditemukan dari direktori kerja ke atas")


ROOT = find_root()
os.chdir(ROOT)

_ENV = {"LARYNX_DATASET": "dataset", "LARYNX_OUTPUTS": "outputs", "LARYNX_HF_CACHE": "hf_cache"}


def deep_merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out


def _load(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = cfg.pop("_base", None)
    return deep_merge(_load(path.parent / base), cfg) if base else cfg


def load_merged(path) -> dict:
    cfg = _load(Path(path))
    for env, key in _ENV.items():
        if os.environ.get(env):
            cfg["paths"][key] = os.environ[env]
    for key, val in cfg["paths"].items():           # jalur relatif -> mutlak terhadap folder Versi 2
        if not Path(val).is_absolute():
            cfg["paths"][key] = str((ROOT / val).resolve())
    return cfg


def P(cfg: dict, *parts: str) -> Path:
    return Path(cfg["paths"]["outputs"]).joinpath(*parts)


def config_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()


def job_seed(base_seed: int, *parts) -> int:
    key = "|".join([str(base_seed), *map(str, parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def atomic_write_bytes(path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def atomic_write_json(path, obj) -> None:
    atomic_write_bytes(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8"))


def atomic_torch_save(obj, path) -> None:
    buf = io.BytesIO(); torch.save(obj, buf); atomic_write_bytes(path, buf.getvalue())


def is_done(job_dir, cfg_hash: str) -> bool:
    done = Path(job_dir) / "DONE"
    return done.exists() and done.read_text(encoding="utf-8").strip() == cfg_hash


def mark_done(job_dir, cfg_hash: str) -> None:
    atomic_write_bytes(Path(job_dir) / "DONE", cfg_hash.encode("utf-8"))


def git_commit() -> str:
    # Versi 2 tanpa Docker: hash commit dari git (bila ada), ditambah -dirty bila ada perubahan belum di-commit.
    try:
        c = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=15).stdout.strip()
        d = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"], capture_output=True, text=True, timeout=15).stdout.strip()
        return (c or "tanpa-commit") + ("-dirty" if d and c else "")
    except Exception:  # noqa: BLE001
        return "tanpa-commit"


GIT_COMMIT = git_commit()
CFG_PATH = "configs/base.yaml"
cfg = load_merged(CFG_PATH)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = P(cfg, "cp0")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT   :", ROOT)
print("dataset:", cfg["paths"]["dataset"])
print("outputs:", cfg["paths"]["outputs"])
print("device :", DEV, "| commit:", GIT_COMMIT, "| config_hash:", config_hash(cfg)[:12])

def free_gpu() -> None:
    # Pengganti 'proses baru per job' di Versi 1: dipanggil di akhir tiap job agar memori GPU kembali bersih.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


WORK_DIR = Path(cfg["paths"]["outputs"]) / "_workers"


def install_module(name: str, src: str):
    # Windows memakai 'spawn': fungsi/kelas yang dipakai proses anak harus berada di modul yang dapat diimpor. Modul dibuat
    # saat notebook berjalan (satu-satunya sumber kodenya ada di sel notebook ini) lalu diimpor, termasuk oleh proses anak.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p = WORK_DIR / f"{name}.py"
    changed = (not p.exists()) or p.read_text(encoding="utf-8") != src
    if changed:
        p.write_text(src, encoding="utf-8")
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod) if changed and name in sys.modules else mod

In [ ]:
# ---------------------------------------------------------------- checkpoint, persetujuan manusia, LOG.md
# Pengganti checkpoint.py dan approve.py Versi 1. Persetujuan MANUSIA tidak pernah dipanggil otomatis oleh notebook.
LULUS, GAGAL, MENUNGGU, TIDAK_TERSEDIA = "LULUS", "GAGAL", "MENUNGGU", "TIDAK_TERSEDIA"
DEC_PATH, BASE_PATH, LOG_PATH = ROOT / "configs" / "decisions.yaml", ROOT / "configs" / "base.yaml", ROOT / "LOG.md"


def file_sha256(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_manifest(out_dir, files, extra=None) -> Path:
    out_dir = Path(out_dir)
    entries = {}
    for f in files:
        p = Path(f)
        if not p.exists():
            raise FileNotFoundError(f"Keluaran checkpoint tidak ditemukan: {p}")
        key = str(p.relative_to(out_dir)) if p.is_relative_to(out_dir) else str(p)
        entries[key] = {"sha256": file_sha256(p), "bytes": p.stat().st_size}
    mp = out_dir / "manifest.json"
    atomic_write_json(mp, {"files": entries, **(extra or {})})
    return mp


def _item(nama: str, status: str, detail: str) -> dict:
    return {"item": nama, "status": status, "detail": detail}


def _review_item(out_dir) -> dict:
    out_dir = Path(out_dir)
    ok = (out_dir / "REVIEWED").exists()
    return _item("Tinjauan orang kedua (berkas REVIEWED)", LULUS if ok else MENUNGGU,
                 "ada" if ok else f"setelah meninjau hasil: approve_review('{out_dir.name.upper()}', 'NAMA') (RULES §3)")


def _flag_item(out_dir, name: str, nama: str, how: str) -> dict:
    ok = (Path(out_dir) / name).exists()
    return _item(nama, LULUS if ok else MENUNGGU, "ada" if ok else how)


def _decision_item(d: str) -> dict:
    with open(DEC_PATH, "r", encoding="utf-8") as f:
        dec = yaml.safe_load(f)[d]
    st = dec["status"]
    status = {"diputuskan": LULUS, "ditunda": LULUS, "menunggu_konfirmasi": MENUNGGU}.get(st, GAGAL)
    return _item(f"{d} {st}", status, str(dec.get("isi", "")))


def _finish(cp: str, out_dir, items: list, files: list, extra=None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    blocking = [i for i in items if i["status"] in (GAGAL, MENUNGGU)]
    rep = {"checkpoint": cp, "lulus": not blocking, "items": items, "penghambat": [i["item"] for i in blocking]}
    atomic_write_json(out_dir / f"{cp.lower()}_check.json", rep)
    for i in items:
        print(f"[{i['status']:>15}] {i['item']} -- {i['detail']}")
    if rep["lulus"]:
        write_manifest(out_dir, [out_dir / f"{cp.lower()}_check.json", *[f for f in files if Path(f).exists()]],
                       extra={"checkpoint": cp, "git_commit": GIT_COMMIT, **(extra or {})})
        print(f"\n{cp} LULUS. manifest.json ditulis. Tag git '{cp.lower()}' dapat dibuat (setelah tinjauan orang kedua).")
    else:
        print(f"\n{cp} BELUM LULUS. Penghambat: {rep['penghambat']}\nmanifest.json TIDAK ditulis.")
    return rep


def log_row(who: str, what: str, hasil: str = "disetujui", keputusan: str = "via notebook") -> None:
    line = f"| {dt.date.today().isoformat()} | {who} | {what} | {hasil} | {keputusan} |\n"
    if LOG_PATH.exists():
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line)


def approve_flag(cp: str, name: str, by: str, what: str) -> None:
    d = P(load_merged(BASE_PATH), cp.lower())
    d.mkdir(parents=True, exist_ok=True)
    (d / name).write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, f"{what} ({cp})")
    print("ditulis", d / name)


def approve_review(cp: str, by: str) -> None:
    approve_flag(cp, "REVIEWED", by, "Tinjauan orang kedua")


def approve_decisions(ids: list, by: str) -> None:
    text = DEC_PATH.read_text(encoding="utf-8")
    for d in ids:
        pat = re.compile(rf"(^{d}:\n  status: )menunggu_konfirmasi", re.M)
        if not pat.search(text):
            print(f"{d}: tidak berstatus menunggu_konfirmasi (dilewati)")
            continue
        text = pat.sub(rf"\1diputuskan\n  dikonfirmasi_oleh: {by}\n  tanggal_konfirmasi: {dt.date.today().isoformat()}", text)
        log_row(by, f"Konfirmasi keputusan {d}")
        print(f"{d}: diputuskan oleh {by}")
    DEC_PATH.write_text(text, encoding="utf-8")


def approve_roi(by: str) -> None:
    t = BASE_PATH.read_text(encoding="utf-8")
    t2 = re.sub(r"(  roi:.*?\n    locked: )false", r"\1true", t, count=1, flags=re.S)
    if t2 == t and "locked: true" not in t:
        raise SystemExit("preprocess.roi.locked tidak ditemukan di base.yaml")
    BASE_PATH.write_text(t2, encoding="utf-8")
    d = P(load_merged(BASE_PATH), "cp2")
    d.mkdir(parents=True, exist_ok=True)
    (d / "ROI_APPROVED").write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, "Parameter Frangi dikunci setelah pemeriksaan visual 50 citra (FR-03)")
    print("preprocess.roi.locked=true; ditulis", d / "ROI_APPROVED")

In [ ]:
# ---------------------------------------------------------------- pengendalian kebocoran L1-L6 (dari leakage.py Versi 1)
class LeakageError(AssertionError):
    pass


def check_l1(splits: dict) -> bool:
    # L1: tidak ada pasien di latih/validasi dan uji sekaligus; tiap pasien diuji tepat sekali per ulangan.
    allp = set(splits["all_patients"])
    for r in splits["repeats"]:
        tested = []
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if tr & te:
                raise LeakageError(f"L1: pasien latih juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(tr & te)[:5]}")
            if va & te:
                raise LeakageError(f"L1: pasien validasi juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(va & te)[:5]}")
            if tr & va:
                raise LeakageError(f"L1: pasien latih juga di validasi (rep {r['rep']} lipatan {f['fold']})")
            if tr | va | te != allp:
                raise LeakageError(f"L1: gabungan partisi != seluruh pasien (rep {r['rep']} lipatan {f['fold']})")
            tested += list(te)
        if sorted(tested) != sorted(allp):
            raise LeakageError(f"L1: pada rep {r['rep']} tiap pasien harus diuji tepat sekali")
    return True


def check_l5(splits: dict, resolution_of: dict) -> bool:
    # L5 (S7): semua pasien latih/validasi berresolusi mayoritas, semua pasien uji bukan.
    major = splits["major_resolution"]
    for r in splits["repeats"]:
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if (tr | va) & te or tr & va:
                raise LeakageError(f"L5: partisi S7 saling tumpang tindih (rep {r['rep']})")
            bad = [p for p in tr | va if resolution_of[p] != major]
            if bad:
                raise LeakageError(f"L5: pasien non-mayoritas di latih/validasi: {sorted(bad)[:5]}")
            bad = [p for p in te if resolution_of[p] == major]
            if bad:
                raise LeakageError(f"L5: pasien resolusi mayoritas di uji S7: {sorted(bad)[:5]}")
    return True


def check_splits(splits: dict, resolution_of=None) -> bool:
    if splits.get("kind") == "s7":
        if resolution_of is None:
            raise LeakageError("L5: peta resolusi pasien diperlukan untuk memeriksa split S7")
        return check_l5(splits, resolution_of)
    return check_l1(splits)


def check_l6(cfg: dict) -> bool:
    # L6: metrik struktur pembuluh memakai filter berbeda dari pembuat ROI.
    if cfg["gan"]["quality"]["vessel_filter"].lower() == cfg["preprocess"]["roi"]["filter"].lower():
        raise LeakageError("L6: filter metrik pembuluh sama dengan filter pembuat ROI")
    return True


def check_l3_job(job_patients: dict, split_fold: dict, synthetic_source_patients=None) -> bool:
    # L3: pasien yang dipakai job sama dengan partisi split; citra sintetis hanya bersumber dari pasien latih.
    test = set(split_fold["test"])
    used = set(job_patients["train"]) | set(job_patients["val"])
    if used & test:
        raise LeakageError(f"L3: job memakai pasien uji: {sorted(used & test)[:5]}")
    if set(job_patients["train"]) != set(split_fold["train"]) or set(job_patients["val"]) != set(split_fold["val"]):
        raise LeakageError("L3: daftar pasien job berbeda dari berkas split")
    if set(job_patients["test"]) != test:
        raise LeakageError("L3: daftar pasien uji job berbeda dari berkas split")
    if synthetic_source_patients is not None and not set(synthetic_source_patients) <= set(split_fold["train"]):
        raise LeakageError("L3: citra sintetis bersumber dari pasien non-latih")
    return True


def check_l3_dir(jobs_root, scenario: str, splits: dict) -> int:
    n = 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            pj = Path(jobs_root) / scenario / f"rep{r['rep']}_fold{f['fold']}" / "patients.json"
            if pj.exists():
                check_l3_job(json.loads(pj.read_text(encoding="utf-8")), f)
                n += 1
    return n


class GuardedArray:
    # L2: pembungkus larik yang menolak akses ke baris terlarang (indeks uji) dan penyerahan seluruh larik.
    def __init__(self, arr, forbidden):
        self._arr = arr
        self._forbidden = np.asarray(sorted(set(map(int, forbidden))), dtype=np.int64)
        self.accessed = set()

    def __len__(self):
        return len(self._arr)

    @property
    def shape(self):
        return self._arr.shape

    def __array__(self, *a, **k):
        raise LeakageError("L2: seluruh larik diminta pada fase fit; gunakan indeks latih/validasi saja")

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            pos = np.arange(len(self._arr))[idx]
        else:
            idx_a = np.asarray(idx)
            pos = np.flatnonzero(idx_a) if idx_a.dtype == bool else idx_a.ravel().astype(np.int64)
        if np.intersect1d(pos, self._forbidden).size:
            raise LeakageError("L2: fase fit mengakses baris data uji")
        self.accessed.update(int(p) for p in pos)
        return np.asarray(self._arr[idx])

In [ ]:
# ---------------------------------------------------------------- keluaran job, pembacaan split, penjalan job (dari jobio.py, splits.py, jobs.py)
def job_dir(cfg: dict, scenario: str, rep: int, fold: int) -> Path:
    return P(cfg, "jobs", scenario, f"rep{rep}_fold{fold}")


def resolution_map(cfg: dict) -> dict:
    a = pd.read_csv(P(cfg, "cp1", "audit.csv"))
    return dict(zip(a.patient_uid, a.resolution))


def write_run_info(d: Path, cfg: dict, cfg_hash: str, seed: int, extra: dict) -> None:
    atomic_write_json(d / "run_info.json", {
        "scenario": cfg.get("scenario"), "config_hash": cfg_hash, "seed": seed, "git_commit": GIT_COMMIT,
        "torch": torch.__version__, "platform": platform.platform(), **extra})


def write_preds(d: Path, name: str, meta: pd.DataFrame, idx: np.ndarray, probs: np.ndarray, thr: float) -> pd.DataFrame:
    # preds_<name>_images.csv (per citra) dan preds_<name>_patients.csv (rata-rata per pasien, FR-09)
    df = pd.DataFrame({"idx": idx, "patient_uid": meta.patient_uid.to_numpy()[idx],
                       "label": meta.label.to_numpy()[idx], "prob_malignant": probs})
    atomic_write_bytes(d / f"preds_{name}_images.csv", df.to_csv(index=False).encode("utf-8"))
    pat = df.groupby("patient_uid").agg(label=("label", "first"), prob_malignant=("prob_malignant", "mean"),
                                        n_images=("idx", "size")).reset_index()
    pat["pred"] = (pat.prob_malignant >= thr).astype(int)
    atomic_write_bytes(d / f"preds_{name}_patients.csv", pat.to_csv(index=False).encode("utf-8"))
    return pat


def balanced_logloss(y: np.ndarray, p1: np.ndarray, eps: float = 1e-7) -> float:
    p1 = np.clip(p1, eps, 1 - eps)
    losses = [(-np.log(1 - p1[y == 0])).mean() if (y == 0).any() else None,
              (-np.log(p1[y == 1])).mean() if (y == 1).any() else None]
    return float(np.mean([v for v in losses if v is not None]))


# ---------------------------------------------------------------- split: baca
def _split_name(splits_name: str):
    return ("splits.json", "fold_summary.csv") if splits_name == "main" else (f"splits_{splits_name}.json", f"fold_summary_{splits_name}.csv")


def load_splits(cfg: dict, splits_name=None) -> dict:
    fj, _ = _split_name(splits_name or cfg.get("splits_name", "main"))
    return json.loads(P(cfg, "splits", fj).read_text(encoding="utf-8"))


def get_fold(splits: dict, rep: int, fold: int) -> dict:
    return splits["repeats"][rep]["folds"][fold]


# ---------------------------------------------------------------- penjalan job (pengganti jobs.py: pemanggilan fungsi, bukan subproses)
def job_status(cfg: dict, scenario: str) -> list:
    rows = []
    for r in load_splits(cfg)["repeats"]:
        for f in r["folds"]:
            d = job_dir(cfg, scenario, r["rep"], f["fold"])
            rows.append({"rep": r["rep"], "fold": f["fold"], "DONE": (d / "DONE").exists(), "ada_last_pt": (d / "last.pt").exists()})
    return rows


def _selected(splits: dict, reps, folds):
    for r in splits["repeats"]:
        if reps is not None and r["rep"] not in reps:
            continue
        for f in r["folds"]:
            if folds is None or f["fold"] in folds:
                yield r["rep"], f["fold"]


def run_job_safe(label: str, fn, cfg: dict, attempts: int = 2):
    """Jalankan satu job. Hanya RuntimeError (termasuk CUDA OOM) diulang sekali setelah memori GPU dibersihkan; job yang gagal
    dua kali menghentikan antrean dan HARUS dicatat di LOG.md (RULES §4). Tidak ada seed diganti atau lipatan dibuang."""
    log = P(cfg, "jobs", "launch.log")
    log.parent.mkdir(parents=True, exist_ok=True)
    err = None
    for a in range(1, attempts + 1):
        t0 = time.time()
        try:
            res = fn()
            ok = True
        except RuntimeError as e:
            res, ok, err = None, False, repr(e)[:300]
        free_gpu()
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(f"{dt.datetime.now().isoformat(timespec='seconds')}\t{label}\tpercobaan={a}\t{'OK' if ok else 'GAGAL ' + str(err)}\t{time.time() - t0:.0f}s\n")
        if ok:
            return res
    raise RuntimeError(f"Job gagal {attempts}x berturut-turut: {label}. Antrean dihentikan. Catat penyebabnya di LOG.md sebelum dilanjutkan (RULES §4). Terakhir: {err}")


def launch_scenario(config: str, fn, reps=None, folds=None) -> None:
    # satu job per (rep, lipatan): fn(cfg, rep, fold)
    cfg = load_merged(config)
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{cfg['scenario']}/rep{r}_fold{k}", lambda r=r, k=k: fn(load_merged(config), r, k), cfg)


def launch_gan_chain(config: str, gan_fn, clf_fn, ratios=None, reps=None, folds=None) -> None:
    # S5/S6/S7: GAN lipatan k, lalu klasifikator tiap rasio (urutan wajib, PRD §8). ratios=[None] untuk S7.
    cfg = load_merged(config)
    sc = cfg["scenario"]
    ratios = ratios if ratios is not None else cfg["gan"]["ratios"]
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{sc}_gan/rep{r}_fold{k}", lambda r=r, k=k: gan_fn(load_merged(config), r, k), cfg)
        for ra in ratios:
            def _clf(r=r, k=k, ra=ra):
                c = load_merged(config)
                if ra is not None:
                    c["synthetic"]["ratio"] = ra
                return clf_fn(c, r, k)
            tag = f"_r{int(round(ra * 100))}" if ra is not None else ""
            run_job_safe(f"{sc}{tag}/rep{r}_fold{k}", _clf, cfg)

In [ ]:
# ---------------------------------------------------------------- pembacaan cache memmap + Dataset (dari data.py Versi 1)
DATA_SRC = r'''
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2


class Cache:
    # images/roi dibuka MALAS dengan memmap dan TIDAK ikut ter-pickle: pickle np.memmap menyalin seluruh isinya
    # (2,2 GB x jumlah pekerja DataLoader di Windows/spawn). Tiap proses membuka ulang memmap sendiri.
    def __init__(self, cache_dir):
        self.dir = Path(cache_dir)
        self.hash = (self.dir / "DONE").read_text(encoding="utf-8").strip()
        self.meta = pd.read_csv(self.dir / "meta.csv")
        self._images = None
        self._roi = None

    @property
    def images(self):
        if self._images is None:
            self._images = np.load(self.dir / "images.npy", mmap_mode="r")
        return self._images

    @property
    def roi(self):
        if self._roi is None:
            self._roi = np.load(self.dir / "roi.npy", mmap_mode="r")
        return self._roi

    def __getstate__(self):
        st = self.__dict__.copy()
        st["_images"] = None
        st["_roi"] = None
        return st

    def indices_of(self, patients):
        return self.meta.index[self.meta.patient_uid.isin(set(patients))].to_numpy()

    @property
    def labels(self):
        return self.meta.label.to_numpy()


def build_transform(cfg, train):
    size, aug, pp = cfg["model"]["input_size"], cfg["augment"], cfg["preprocess"]
    t = []
    if train:
        t.append(v2.RandomResizedCrop(size, scale=tuple(aug["crop_scale"]), ratio=tuple(aug["crop_ratio"]), antialias=True))
        if aug["hflip"]:
            t.append(v2.RandomHorizontalFlip())
        if aug["vflip"]:
            t.append(v2.RandomVerticalFlip())
        if aug["rotation_deg"]:
            t.append(v2.RandomRotation(aug["rotation_deg"]))
    else:
        t.append(v2.Resize((size, size), antialias=True))
    t += [v2.ToDtype(torch.float32, scale=True), v2.Normalize(pp["imagenet_mean"], pp["imagenet_std"])]
    return v2.Compose(t)


class ImageDS(Dataset):
    # (tensor ternormalisasi ImageNet, label, indeks cache), dilabeli ganas dan indeks -1 untuk sintetis.
    # synthetic: None | array uint8 (N,H,W,3) | (jalur .npy, indeks terpilih): bentuk terakhir dibuka dengan memmap di tiap
    # proses (tidak ikut ter-pickle ke pekerja DataLoader).
    def __init__(self, cache, idx, cfg, train, synthetic=None):
        self.cache, self.idx = cache, np.asarray(idx)
        self.syn, self.syn_path, self.syn_idx, self._syn_mm = None, None, None, None
        if isinstance(synthetic, tuple):
            self.syn_path, self.syn_idx = str(synthetic[0]), np.asarray(synthetic[1])
            n_syn = len(self.syn_idx)
        elif synthetic is not None:
            self.syn = synthetic
            n_syn = len(synthetic)
        else:
            n_syn = 0
        self.n_real = len(self.idx)
        self.tf = build_transform(cfg, train)
        self.labels = np.concatenate([cache.labels[self.idx], np.ones(n_syn, dtype=cache.labels.dtype)])

    def __getstate__(self):
        st = self.__dict__.copy()
        st["_syn_mm"] = None
        return st

    def _synthetic(self, k):
        if self.syn_path is None:
            return self.syn[k]
        if self._syn_mm is None:
            self._syn_mm = np.load(self.syn_path, mmap_mode="r")
        return self._syn_mm[int(self.syn_idx[k])]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        if i < self.n_real:
            j = int(self.idx[i])
            arr = np.array(self.cache.images[j])
        else:
            j = -1
            arr = np.array(self._synthetic(i - self.n_real))
        x = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
        return self.tf(x), int(self.labels[i]), j
'''

larynx_data = install_module("larynx_data", DATA_SRC)
Cache, ImageDS = larynx_data.Cache, larynx_data.ImageDS


def find_cache(cfg: dict) -> Path:
    cur = P(cfg, "cache", "current.json")
    if not cur.exists():
        raise FileNotFoundError("Cache belum dibuat (jalankan notebook 02)")
    return Path(json.loads(cur.read_text(encoding="utf-8"))["dir"])

## 3. Kode praproses dan cache (dari `preprocess.py`)

Fungsi per-citra ditulis ke modul `outputs/_workers/larynx_pp.py` saat sel ini dijalankan, supaya proses anak (spawn, Windows) dapat mengimpornya.

In [ ]:
# ---------------------------------------------------------------- praproses lima langkah + cache memmap (dari preprocess.py Versi 1)
# Fungsi per-citra berada di modul 'larynx_pp' agar dapat dipakai proses anak (spawn, Windows).
PP_SRC = r'''
from pathlib import Path

import numpy as np
from PIL import Image
from scipy import ndimage as ndi
from skimage.filters import frangi, gaussian


def fov_mask(rgb, thr, min_area_fraction):
    gray = rgb.mean(axis=2)
    m = ndi.binary_fill_holes(gray > thr)
    lab, n = ndi.label(m)
    if n > 1:
        sizes = ndi.sum(m, lab, index=np.arange(1, n + 1))
        m = lab == (1 + int(np.argmax(sizes)))
    if m.mean() < min_area_fraction:
        return np.ones(gray.shape, bool), True       # deteksi gagal -> seluruh citra, ditandai
    return m, False


def crop_square(rgb, mask):
    ys, xs = np.where(mask)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    img = (rgb * mask[..., None])[y0:y1, x0:x1]
    msk = mask[y0:y1, x0:x1]
    h, w = msk.shape
    s = max(h, w)
    pad_img = np.zeros((s, s, 3), np.uint8)
    pad_msk = np.zeros((s, s), bool)
    oy, ox = (s - h) // 2, (s - w) // 2
    pad_img[oy:oy + h, ox:ox + w] = img
    pad_msk[oy:oy + h, ox:ox + w] = msk
    return pad_img, pad_msk


def _remove_small(m, min_size):
    lab, n = ndi.label(m)
    if n == 0:
        return m
    sizes = np.bincount(lab.ravel())
    keep = sizes >= min_size
    keep[0] = False
    return keep[lab]


def roi_map(img, fov, r):
    fe = ndi.binary_erosion(fov, iterations=int(r["fov_erode"]))
    if fe.sum() < 100:
        fe = fov
    g = img[..., 1].astype(np.float32) / 255.0
    g = np.where(fe, g, g[fe].mean() if fe.any() else 0.0)
    g = gaussian(g, r["green_blur"])
    v = frangi(g, sigmas=tuple(r["sigmas"]), alpha=r["alpha"], beta=r["beta"], gamma=r["gamma"], black_ridges=True)
    v = v * fe
    v = np.clip(v / (np.percentile(v[fe], r["norm_percentile"]) + 1e-8), 0, 1) if fe.any() else v
    m = (v > r["threshold"]) & fe
    return _remove_small(m, int(r["min_obj"])).astype(np.uint8)


def process_array(rgb, pp):
    S = pp["cache_size"]
    mask, fallback = fov_mask(rgb, pp["fov"]["threshold"], pp["fov"]["min_area_fraction"])
    sq, sqm = crop_square(rgb, mask)
    img = np.asarray(Image.fromarray(sq).resize((S, S), Image.LANCZOS), np.uint8)
    fov = np.asarray(Image.fromarray(sqm.astype(np.uint8) * 255).resize((S, S), Image.BILINEAR)) > 127
    gray = img.mean(axis=2)
    q = pp["quality"]
    inside = fov if fov.any() else np.ones_like(fov)
    lap = ndi.laplace(gray)
    quality = {"q_blur_lapvar": float(lap[inside].var()), "q_brightness": float(gray[inside].mean()),
               "q_glare_frac": float(((img.max(axis=2) >= q["saturation_level"]) & inside).mean()),
               "q_dark_frac": float((gray[inside] < q["dark_level"]).mean()), "q_fov_frac": float(fov.mean()),
               "fov_fallback": bool(fallback)}
    return {"img": img, "roi": roi_map(img, fov, pp["roi"]), "quality": quality}


def process_file(path, pp):
    with Image.open(path) as im:
        rgb = np.asarray(im.convert("RGB"), np.uint8)
    return process_array(rgb, pp)


_PP = {}
_ROOT = None


def _init(pp, root):
    global _PP, _ROOT
    _PP, _ROOT = pp, Path(root)


def _work(rel):
    return process_file(_ROOT / rel, _PP)
'''

larynx_pp = install_module("larynx_pp", PP_SRC)


def cache_hash(cfg: dict, images: pd.DataFrame) -> str:
    pp = {k: v for k, v in cfg["preprocess"].items() if k not in ("workers", "imagenet_mean", "imagenet_std", "normalize")}
    pp["roi"] = {k: v for k, v in pp["roi"].items() if k != "locked"}      # mengunci parameter tidak memaksa cache dibangun ulang
    pp["roi_examples"] = None
    return config_hash({"pp": pp, "files": list(zip(images.rel_path, images.bytes))})


def build_cache(cfg: dict, images=None, workers=None) -> Path:
    # Dijalankan sekali. Bila cache dengan hash sama sudah DONE, dilewati.
    if images is None:
        images = pd.read_csv(P(cfg, "cp1", "audit_images.csv"))
    images = images.reset_index(drop=True)
    h = cache_hash(cfg, images)
    root = P(cfg, "cache")
    final = root / h[:12]
    if final.exists() and is_done(final, h):
        atomic_write_json(root / "current.json", {"hash": h, "dir": str(final)})
        print(f"[dilewati] cache {final.name} sudah DONE")
        return final
    tmp = root / (h[:12] + ".partial")
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True)
    N, S = len(images), cfg["preprocess"]["cache_size"]
    imgs = np.lib.format.open_memmap(tmp / "images.npy", mode="w+", dtype=np.uint8, shape=(N, S, S, 3))
    rois = np.lib.format.open_memmap(tmp / "roi.npy", mode="w+", dtype=np.uint8, shape=(N, S, S))
    qual = []
    workers = workers or cfg["preprocess"]["workers"]
    rels = images.rel_path.tolist()
    if workers > 1:
        import multiprocessing as mp
        pool = mp.get_context("spawn").Pool(workers, initializer=larynx_pp._init, initargs=(cfg["preprocess"], cfg["paths"]["dataset"]))
        it = pool.imap(larynx_pp._work, rels, chunksize=8)
    else:
        larynx_pp._init(cfg["preprocess"], cfg["paths"]["dataset"])
        pool, it = None, map(larynx_pp._work, rels)
    for i, r in enumerate(it):
        imgs[i], rois[i] = r["img"], r["roi"]
        qual.append(r["quality"])
        if (i + 1) % 500 == 0:
            print(f"  praproses {i + 1}/{N}", flush=True)
    if pool is not None:
        pool.close(); pool.join()
    imgs.flush(); rois.flush()
    del imgs, rois
    meta = pd.concat([images.reset_index(drop=True), pd.DataFrame(qual)], axis=1)
    meta.insert(0, "idx", np.arange(N))
    (tmp / "meta.csv").write_text(meta.to_csv(index=False), encoding="utf-8")
    if final.exists():
        shutil.rmtree(final)
    os.replace(tmp, final)
    mark_done(final, h)
    atomic_write_json(root / "current.json", {"hash": h, "dir": str(final)})
    return final


def save_roi_examples(cfg: dict, cache_dir, out_dir) -> Path:
    import matplotlib
    import matplotlib.pyplot as plt

    cache_dir, out_dir = Path(cache_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    meta = pd.read_csv(cache_dir / "meta.csv")
    imgs = np.load(cache_dir / "images.npy", mmap_mode="r")
    rois = np.load(cache_dir / "roi.npy", mmap_mode="r")
    ex = cfg["preprocess"]["roi_examples"]
    rng = np.random.default_rng(ex["seed"])
    pick = np.sort(rng.choice(len(meta), size=min(ex["n"], len(meta)), replace=False))
    rows = meta.iloc[pick][["idx", "rel_path", "patient_uid", "label", "class_name"]].copy()
    rows["roi_fraction"] = [float(rois[i].mean()) for i in pick]
    atomic_write_bytes(out_dir / "roi_examples_patients.csv", rows.to_csv(index=False).encode("utf-8"))      # D6
    per_page = 10
    for page in range(0, len(pick), per_page):
        chunk = pick[page:page + per_page]
        fig, axes = plt.subplots(len(chunk), 3, figsize=(7.5, 2.5 * len(chunk)), squeeze=False)
        for ax_row, i in zip(axes, chunk):
            ov = np.asarray(imgs[i]).astype(np.float32).copy()
            m = np.asarray(rois[i]).astype(bool)
            ov[m] = 0.35 * ov[m] + 0.65 * np.array([255, 255, 0], np.float32)
            ax_row[0].imshow(imgs[i]); ax_row[1].imshow(rois[i], cmap="gray", vmin=0, vmax=1); ax_row[2].imshow(ov.astype(np.uint8))
            ax_row[0].set_ylabel(f"{meta.patient_uid[i]}", fontsize=7)
            for a in ax_row:
                a.set_xticks([]); a.set_yticks([])
        axes[0][0].set_title("citra 256", fontsize=8); axes[0][1].set_title("ROI Frangi", fontsize=8); axes[0][2].set_title("overlay", fontsize=8)
        fig.tight_layout()
        fig.savefig(out_dir / f"roi_check_{page // per_page + 1}.png", dpi=110)
        plt.close(fig)
    return out_dir

## 4. Kode pembuatan split (dari `splits.py`)

In [ ]:
# ---------------------------------------------------------------- pembuatan split per pasien (dari splits.py Versi 1, FR-04)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split


def _patient_table(patients: pd.DataFrame) -> pd.DataFrame:
    return (patients[["patient_uid", "label", "n_images"]].drop_duplicates("patient_uid").sort_values("patient_uid").reset_index(drop=True))


def make_splits(patients: pd.DataFrame, cfg: dict) -> dict:
    sp, seed = cfg["splits"], cfg["seed"]
    pt = _patient_table(patients)
    uids, y = pt.patient_uid.to_numpy(), pt.label.to_numpy()
    repeats = []
    for rep in range(sp["n_repeats"]):
        skf = StratifiedGroupKFold(n_splits=sp["n_splits"], shuffle=True, random_state=job_seed(seed, "split", rep))
        folds = []
        for k, (tr_all, te) in enumerate(skf.split(np.zeros(len(uids)), y, groups=uids)):
            tr, va = train_test_split(tr_all, test_size=sp["val_fraction"], stratify=y[tr_all], random_state=job_seed(seed, "val", rep, k))
            folds.append({"fold": k, "train": sorted(uids[tr]), "val": sorted(uids[va]), "test": sorted(uids[te])})
        repeats.append({"rep": rep, "folds": folds})
    splits = {"kind": "main", "all_patients": sorted(uids), "repeats": repeats,
              "meta": {"seed": seed, "n_splits": sp["n_splits"], "n_repeats": sp["n_repeats"], "val_fraction": sp["val_fraction"],
                       "patient_table_hash": config_hash(pt.astype(str).to_dict(orient="list"))}}
    splits["meta"]["hash"] = config_hash(splits)
    check_l1(splits)
    return splits


def make_s7_splits(patients: pd.DataFrame, cfg: dict) -> dict:
    sp, seed, s7 = cfg["splits"], cfg["seed"], cfg["s7"]
    if (patients.n_resolutions > 1).any():
        raise ValueError("Ada pasien dengan >1 resolusi; S7 tidak dapat menetapkan kelompok (lihat audit)")
    major = patients.resolution.value_counts().index[0]
    tr_pool = patients[patients.resolution == major]
    test = patients[patients.resolution != major]
    pt = _patient_table(tr_pool)
    repeats = []
    for rep in range(s7["n_repeats"]):
        tr, va = train_test_split(np.arange(len(pt)), test_size=sp["val_fraction"], stratify=pt.label.to_numpy(),
                                  random_state=job_seed(seed, "s7val", rep))
        repeats.append({"rep": rep, "folds": [{"fold": 0, "train": sorted(pt.patient_uid.to_numpy()[tr]),
                                               "val": sorted(pt.patient_uid.to_numpy()[va]), "test": sorted(test.patient_uid)}]})
    splits = {"kind": "s7", "all_patients": sorted(patients.patient_uid), "major_resolution": major,
              "test_resolutions": sorted(test.resolution.unique().tolist()), "repeats": repeats,
              "meta": {"seed": seed, "n_repeats": s7["n_repeats"], "val_fraction": sp["val_fraction"],
                       "patient_table_hash": config_hash(_patient_table(patients).astype(str).to_dict(orient="list"))}}
    splits["meta"]["hash"] = config_hash(splits)
    check_l5(splits, dict(zip(patients.patient_uid, patients.resolution)))
    return splits


def fold_summary(splits: dict, patients: pd.DataFrame) -> pd.DataFrame:
    lab = dict(zip(patients.patient_uid, patients.label))
    nimg = dict(zip(patients.patient_uid, patients.n_images))
    rows = []
    for r in splits["repeats"]:
        for f in r["folds"]:
            row = {"rep": r["rep"], "fold": f["fold"]}
            for part in ("train", "val", "test"):
                ids = f[part]
                row[f"{part}_pasien"] = len(ids)
                row[f"{part}_ganas"] = int(sum(lab[i] for i in ids))
                row[f"{part}_citra"] = int(sum(nimg[i] for i in ids))
            rows.append(row)
    return pd.DataFrame(rows)


def save_splits(cfg: dict, splits: dict, patients: pd.DataFrame, splits_name: str = "main") -> Path:
    d, (fj, fc) = P(cfg, "splits"), _split_name(splits_name)
    atomic_write_json(d / fj, splits)
    atomic_write_bytes(d / fc, fold_summary(splits, patients).to_csv(index=False).encode("utf-8"))
    return d / fj

## 5. Praproses dan cache (idempoten; dilewati bila cache ber-hash sama sudah DONE)

`preprocess.workers` di `configs/base.yaml` menentukan jumlah proses (default 2; naikkan sesuai CPU).

In [ ]:
cache_dir = build_cache(cfg)
save_roi_examples(cfg, cache_dir, P(cfg, 'cp2', 'roi_check'))
print('cache:', cache_dir)

## 6. Split per pasien (dan S7 bila audit menyatakan memenuhi syarat)

In [ ]:
pt = pd.read_csv(P(cfg, 'cp1', 'audit.csv'))
s_main = make_splits(pt, cfg)
save_splits(cfg, s_main, pt, 'main')
display(fold_summary(s_main, pt))
s7_ok = json.loads(P(cfg, 'cp1', 'audit_summary.json').read_text(encoding='utf-8'))['s7'].get('s7_dapat_dijalankan')
print('S7 dapat dijalankan:', s7_ok)
if s7_ok:
    s_s7 = make_s7_splits(pt, cfg)
    save_splits(cfg, s_s7, pt, 's7')
    print('S7: latih', s_s7['major_resolution'], '| uji', s_s7['test_resolutions'])
    display(fold_summary(s_s7, pt))

## 7. Uji kebocoran otomatis (data tiruan; pengganti pytest Versi 1)

Setiap kebocoran buatan harus **ditolak**. Bila ada yang lolos, sel ini gagal dan CP2 tidak dapat lulus.

In [ ]:
def selftest_leakage():
    # Uji otomatis pengganti pytest Versi 1: data tiruan; setiap kebocoran buatan HARUS ditolak.
    out = []

    def must_fail(nama, fn):
        try:
            fn()
        except LeakageError:
            out.append(f"{nama}: ditolak (benar)")
            return
        raise AssertionError(f"{nama}: kebocoran TIDAK terdeteksi")

    arr = np.arange(40).reshape(20, 2)
    g = GuardedArray(arr, forbidden=[17, 18, 19])
    assert g[np.array([1, 2, 3])].shape == (3, 2) and g.accessed == {1, 2, 3}
    must_fail("L2 akses baris uji", lambda: g[np.array([1, 18])])
    must_fail("L2 penyerahan seluruh larik", lambda: np.asarray(g))
    bocor = {"all_patients": ["a", "b", "c"], "repeats": [{"rep": 0, "folds": [
        {"fold": 0, "train": ["a", "b"], "val": [], "test": ["b", "c"]}]}]}
    must_fail("L1 pasien latih = uji", lambda: check_l1(bocor))
    bagus = {"all_patients": ["a", "b", "c"], "repeats": [{"rep": 0, "folds": [
        {"fold": 0, "train": ["a"], "val": [], "test": ["b", "c"]}, {"fold": 1, "train": ["b", "c"], "val": [], "test": ["a"]}]}]}
    assert check_l1(bagus)
    must_fail("L3 pasien uji dipakai job", lambda: check_l3_job({"train": ["a", "b"], "val": [], "test": ["c"]},
                                                                  {"train": ["a"], "val": [], "test": ["b", "c"]}))
    must_fail("L3 sintetis dari pasien non-latih", lambda: check_l3_job({"train": ["a"], "val": [], "test": ["b", "c"]},
                                                                         {"train": ["a"], "val": [], "test": ["b", "c"]}, ["a", "b"]))
    s7_bocor = {"major_resolution": "R1", "repeats": [{"rep": 0, "folds": [{"fold": 0, "train": ["a"], "val": [], "test": ["b"]}]}]}
    must_fail("L5 pasien uji bermayoritas", lambda: check_l5(s7_bocor, {"a": "R1", "b": "R1"}))
    c6 = copy.deepcopy(cfg)
    c6["gan"]["quality"]["vessel_filter"] = c6["preprocess"]["roi"]["filter"]
    must_fail("L6 filter pembuluh sama dengan pembuat ROI", lambda: check_l6(c6))
    return True, "; ".join(out)

SELFTEST = selftest_leakage()
print(SELFTEST)

## 8. Pemeriksaan visual ROI (FR-03)

Periksa gambar. Bila ROI wajar, jalankan sel persetujuan di bawah (mengunci parameter dan menandai). Bila tidak, ubah parameter di `configs/base.yaml` lalu ulangi sel 5; **jangan** memilih parameter berdasarkan hasil klasifikasi.

In [ ]:
d = P(cfg, 'cp2', 'roi_check')
for f in sorted(d.glob('roi_check_*.png'))[:2]:
    display(Image(filename=str(f)))
print('Semua halaman ada di', d)
display(pd.read_csv(d / 'roi_examples_patients.csv').head(10))

In [ ]:
# Setelah memeriksa SEMUA halaman roi_check_*.png: isi nama lalu jalankan. Ini mengubah preprocess.roi.locked menjadi true di base.yaml.
ROI_APPROVER = None      # contoh: 'Wahyu'
if ROI_APPROVER:
    approve_roi(ROI_APPROVER)

## 9. Pemeriksaan checkpoint CP2 dan `manifest.json` (sel terakhir)

In [ ]:
# Isi nama peninjau KEDUA setelah benar-benar meninjau hasil di atas (RULES §3), lalu jalankan sel ini.
REVIEWER = None      # contoh: 'Ahmad'
if REVIEWER:
    approve_review('CP2', REVIEWER)

In [ ]:
def finalize_cp2(cfg: dict, selftest=(False, "uji kebocoran otomatis belum dijalankan")) -> dict:
    cfg = load_merged(BASE_PATH)        # muat ulang: approve_roi() mengubah base.yaml
    out = P(cfg, "cp2")
    out.mkdir(parents=True, exist_ok=True)
    items, files, cache = [], [], None
    try:
        cache = Cache(find_cache(cfg))
        imgs = pd.read_csv(P(cfg, "cp1", "audit_images.csv"))
        items.append(_item("Cache memmap uint8 ber-hash konfigurasi dan sesuai berkas audit",
                           LULUS if cache.hash == cache_hash(cfg, imgs) else GAGAL,
                           f"hash={cache.hash[:12]}, {len(cache.meta)} citra, images{cache.images.shape} roi{cache.roi.shape}"))
        items.append(_item("Jumlah citra cache = audit", LULUS if len(cache.meta) == len(imgs) else GAGAL, f"{len(cache.meta)} vs {len(imgs)}"))
        items.append(_item("Tipe data uint8", LULUS if cache.images.dtype == np.uint8 and cache.roi.dtype == np.uint8 else GAGAL,
                           f"{cache.images.dtype}/{cache.roi.dtype}"))
        q = cache.meta
        items.append(_item("Ringkasan kualitas (informasi, tidak ada citra dibuang)", LULUS,
                           f"fov_fallback={int(q.fov_fallback.sum())}, glare>5%={int((q.q_glare_frac > .05).sum())}, "
                           f"gelap>50%={int((q.q_dark_frac > .5).sum())}, blur terendah 1%={q.q_blur_lapvar.quantile(.01):.1f}"))
        files.append(cache.dir / "meta.csv")
    except Exception as e:  # noqa: BLE001
        items.append(_item("Cache", GAGAL, repr(e)[:200]))
    sp = P(cfg, "splits", "splits.json")
    if sp.exists():
        splits = json.loads(sp.read_text(encoding="utf-8"))
        try:
            check_l1(splits)
            items.append(_item("L1: tidak ada pasien di latih dan uji sekaligus; tiap pasien diuji sekali per ulangan", LULUS,
                               f"{len(splits['repeats'])} ulangan x {len(splits['repeats'][0]['folds'])} lipatan"))
        except LeakageError as e:
            items.append(_item("L1", GAGAL, str(e)))
        try:
            n = check_l3_dir(P(cfg, "jobs"), "S1", splits)
            items.append(_item("L3: daftar pasien job vs split", LULUS, f"{n} job diperiksa (0 wajar pada CP2)"))
        except LeakageError as e:
            items.append(_item("L3", GAGAL, str(e)))
        fs = pd.read_csv(P(cfg, "splits", "fold_summary.csv"))
        items.append(_item("Proporsi pasien ganas per lipatan uji (informasi)", LULUS if (fs.test_ganas > 0).all() else GAGAL,
                           f"uji: pasien {fs.test_pasien.min()}-{fs.test_pasien.max()}, ganas {fs.test_ganas.min()}-{fs.test_ganas.max()}; "
                           f"validasi ganas {fs.val_ganas.min()}-{fs.val_ganas.max()}"))
        files += [sp, P(cfg, "splits", "fold_summary.csv")]
    else:
        items.append(_item("Berkas split", GAGAL, "splits.json tidak ada"))
    s7p = P(cfg, "splits", "splits_s7.json")
    if s7p.exists():
        try:
            s7 = json.loads(s7p.read_text(encoding="utf-8"))
            check_l5(s7, resolution_map(cfg))
            items.append(_item("L5: split S7 (latih resolusi mayoritas, uji gabungan lain)", LULUS, f"latih {s7['major_resolution']}, uji {s7['test_resolutions']}"))
            files.append(s7p)
        except Exception as e:  # noqa: BLE001
            items.append(_item("L5 split S7", GAGAL, repr(e)[:200]))
    else:
        s = json.loads(P(cfg, "cp1", "audit_summary.json").read_text(encoding="utf-8"))["s7"]
        items.append(_item("Split S7", LULUS if not s.get("s7_dapat_dijalankan") else GAGAL,
                           "S7 tidak memenuhi syarat menurut audit" if not s.get("s7_dapat_dijalankan") else "splits_s7.json belum dibuat"))
    items.append(_item("L2 dan uji kebocoran otomatis (uji di notebook, data tiruan)", LULUS if selftest[0] else GAGAL, selftest[1]))
    rc = out / "roi_check"
    pngs = sorted(rc.glob("roi_check_*.png")) if rc.exists() else []
    items.append(_item("50 citra ROI untuk pemeriksaan visual dibuat (FR-03)", LULUS if len(pngs) >= 5 else GAGAL, f"{len(pngs)} halaman di {rc}"))
    items.append(_item("D6: ID pasien 50 citra penyetel Frangi tercatat", LULUS if (rc / "roi_examples_patients.csv").exists() else GAGAL,
                       "roi_examples_patients.csv (tulis sebagai keterbatasan: semua pasien menjadi pasien uji di satu lipatan)"))
    items.append(_decision_item("D6"))
    approved = (out / "ROI_APPROVED").exists() and cfg["preprocess"]["roi"]["locked"]
    items.append(_item("Parameter Frangi dikunci setelah pemeriksaan visual", LULUS if approved else MENUNGGU,
                       "dikunci" if approved else "periksa PNG, lalu: approve_roi('NAMA')"))
    items.append(_review_item(out))
    return _finish("CP2", out, items, files, {"cache_hash": cache.hash if cache is not None else None})


report = finalize_cp2(cfg, SELFTEST)
assert report['checkpoint'] == 'CP2'